<a href="https://colab.research.google.com/github/nehaprasad1/proj1/blob/main/nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

neural network from scratch

In [84]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import math as m

In [85]:

# --- ACTIVATION FUNCTIONS LAYER ---
class ActivationLayer:
    def __init__(self, activation_fn="sigmoid"):
        self.activation_fn = activation_fn.lower()
        self.input = None
        self.output = None

    def activation(self, x):
        self.input = x

        if self.activation_fn == "sigmoid":
            self.output = 1 / (1 + np.exp(-x))
        elif self.activation_fn == "relu":
            self.output = np.maximum(0, x)
        elif self.activation_fn == "tanh":
            self.output = (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))
        elif self.activation_fn == "softmax":
            # Stable Softmax: subtracting np.max prevents exponential overflow (NaN errors)
            shift_x = x - np.max(x, axis=-1, keepdims=True)
            exps = np.exp(shift_x)
            self.output = exps / np.sum(exps, axis=-1, keepdims=True)
        else:
            raise ValueError("Invalid activation function")

        return self.output

    def backprop(self, op_grad):
        """
        BACKPROPAGATION FOR ACTIVATIONS
        """
        if self.activation_fn == "relu":
            return op_grad * (self.input > 0)

        elif self.activation_fn == "sigmoid":
            return op_grad * (self.output * (1 - self.output))

        elif self.activation_fn == "tanh":
            return op_grad * (1 - self.output ** 2)

        elif self.activation_fn == "softmax":
            # Note: This elegant shortcut handles Softmax paired with Categorical Cross-Entropy perfectly.
            # If you combine them, their combined derivative simplifies directly to (y_pred - y_true).
            return op_grad

In [86]:
#activation function
#softmax , relu , tanh ,  sigmoid

'''
def sigmoid(x):
  return 1/(1+np.exp(-x))
def reLU(x):
  return max(0,x)
def tanh(x):
  return (np.exp(x)-np.exp(-x))/(np.exp(x)+np.exp(-x))


class ActivationLayer:
  def __init__(self,activation_fn="sigmoid"):

    self.activation_fn = activation_fn.lower()
    self.input = None
    self.output = None

  def activation(self,x):

    self.input = x

    #simoid function
    if self.activation_fn == "sigmoid":
      self.output=  1/(1+np.exp(-x))

    #relu activation function
    elif self.activation_fn == "relu":
      self.output = np.maximum(0,x)

    #tanh activation function
    elif self.activation_fn == "tanh":
      self.output = (np.exp(x)-np.exp(-x))/(np.exp(x)+np.exp(-x))

    # softmax
    elif self.activation_fn == "softmax":
      self.output = np.exp(x)/np.sum(np.exp(x))

    # if invalid activation function
    else:
      raise ValueError("Invalid activation function")

    return self.output
    '''


'\ndef sigmoid(x):\n  return 1/(1+np.exp(-x))\ndef reLU(x):\n  return max(0,x)\ndef tanh(x):\n  return (np.exp(x)-np.exp(-x))/(np.exp(x)+np.exp(-x))\n\n\nclass ActivationLayer:\n  def __init__(self,activation_fn="sigmoid"):\n\n    self.activation_fn = activation_fn.lower()\n    self.input = None\n    self.output = None\n\n  def activation(self,x):\n\n    self.input = x\n\n    #simoid function\n    if self.activation_fn == "sigmoid":\n      self.output=  1/(1+np.exp(-x))\n\n    #relu activation function\n    elif self.activation_fn == "relu":\n      self.output = np.maximum(0,x)\n\n    #tanh activation function\n    elif self.activation_fn == "tanh":\n      self.output = (np.exp(x)-np.exp(-x))/(np.exp(x)+np.exp(-x))\n\n    # softmax\n    elif self.activation_fn == "softmax":\n      self.output = np.exp(x)/np.sum(np.exp(x))\n\n    # if invalid activation function \n    else:\n      raise ValueError("Invalid activation function")\n\n    return self.output\n    '

In [87]:
#forward Pass and backpropagation
class DenseLayer:
  def __init__(self, ip_dim,op_dim,r=0.01):

    #weight and biass initialization for forward pass
    self.weights = np.random.randn(ip_dim,op_dim)*0.01
    self.bias = np.zeros((1,op_dim))
    self.r = r
    #placeholder
    self.input = None
    self.output = None

    #back-prop placeholders
    self.mod_W = None
    self.mode_B = None

  #forward pass function (z=X.W + b)
  def forwardpass(self,ip):
    self.input = ip
    self.output = np.dot(ip,self.weights) + self.bias

    return self.output

  # back propagation function
  def backprop(self, op_grad):
        # 1. Gradient calculation
        self.mod_W = np.dot(self.input.T, op_grad)
        self.mode_B = np.sum(op_grad, axis=0, keepdims=True)

        # 2. Input gradient calculation to pass backward down the chain
        ip_grad = np.dot(op_grad, self.weights.T)

        # 3. FIX: Update weights and biases BEFORE returning!
        self.weights -= self.r * self.mod_W
        self.bias -= self.r * self.mode_B

        return ip_grad
  '''
  def backprop(self,op_grad,r):

    #gradient calculation dW = X^T . dY
    self.mod_W = np.dot(self.input.T,op_grad)
    self.mode_B = np.sum(op_grad,axis=0,keepdims=True)

    ip_grad = np.dot(op_grad,self.weights.T)
    #return ip_grad

    #update weights and bias
    self.weights -= r*self.mod_W
    self.bias -= r*self.mode_B

    return ip_grad
'''

In [91]:
#loss functions
# mse , categorical cross entropy , log loss
class Loss_fn:
  def __init__(self,loss_fn="mse"):
    self.loss_fn = loss_fn.lower()

  def loss(self,y_true,y_pred):
      y_pred = np.clip(y_pred,1e-15,1-1e-15)

      #mean square error loss function
      if self.loss_fn == "mse":
        L = np.mean(np.power(y_true-y_pred,2))
        return L

      # categorical cross entropy
      #actuall op should be one hot encoded
      elif self.loss_fn == "catxentropy":
        L = -(np.sum(y_true*np.log(y_pred)) / y_pred.shape[0])
        return L

      #log loss function
      elif self.loss_fn == "logloss":
        L = -np.mean(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))
        return L

      #raise excception
      else:
        raise ValueError(f"unknown loss function")
  def backward(self, y_true, y_pred):
        """BACKWARD PASS: Kick off gradient calculations"""
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        m = y_pred.shape[0]

        if self.loss_fn == "mse":
            return 2 * (y_pred - y_true) / y_pred.size
        elif self.loss_fn == "logloss":
            return ((y_pred - y_true) / (y_pred * (1 - y_pred))) / m
        elif self.loss_fn == "catxentropy":
            # For Softmax + CatXEntropy combination, the derivative simplifies beautifully
            return (y_pred - y_true) / m


In [92]:
class NeuralNetwork:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, x):
        out = x
        for layer in self.layers:
            if hasattr(layer, 'forwardpass'):
                out = layer.forwardpass(out)
            else:
                out = layer.activation(out)
        return out

    def backward(self, loss_grad):
        grad = loss_grad
        for layer in reversed(self.layers):
            if hasattr(layer, 'backprop'):
                grad = layer.backward(grad) if hasattr(layer, 'backward') else layer.backprop(grad)
        return grad
'''class NeuralNetwork:
  def __init__(self):
    self.layers = []
    # self.loss = None

  def add(self,layer):
    self.layers.append(layer)

  def forward(self, x):
    out = x
    for layer in self.layers:
      if hasattr(layer, 'forwardpass'):
        out = layer.forwardpass(out)
      else:
        out = layer.activation(out)
    return out

  def backward(self, loss_grad):
    grad = loss_grad
    for layer in reversed(self.layers):
      if hasattr(layer, 'backprop'):
        grad = layer.backprop(grad)
    return grad
 '''

"class NeuralNetwork:\n  def __init__(self):\n    self.layers = []\n    # self.loss = None\n\n  def add(self,layer):\n    self.layers.append(layer)\n\n  def forward(self, x):\n    out = x\n    for layer in self.layers:\n      if hasattr(layer, 'forwardpass'):\n        out = layer.forwardpass(out)\n      else:\n        out = layer.activation(out)\n    return out\n  \n  def backward(self, loss_grad):\n    grad = loss_grad\n    for layer in reversed(self.layers):\n      if hasattr(layer, 'backprop'):\n        grad = layer.backprop(grad)\n    return grad\n "

In [94]:
import os
import urllib.request